# Setup


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
from brighteyes_flim import mcs
from brighteyes_flim import flism
from brighteyes_flim.flism import Alignment

print(mcs.__file__)
print(flism.__file__)


# Helper Functions


In [ ]:
IRF_from_data_deconvolution = Alignment.IRF_from_data_deconvolution
fit_model_data = Alignment.fit_model_data
hist_for_plot = Alignment.hist_for_plot
perform_fit_data = Alignment.perform_fit_data
phasor_delay_from_hist = Alignment.phasor_delay_from_hist
to_numpy_1d = Alignment.to_numpy_1d

# Fixed notebook choices for readability.
# The Alignment class now uses a single public dT convention throughout:
# - binned periodic exponential model
# - circular fit
# - dT reported in [0, nbin)
# - positive dT follows the sign of a positive reference / IRF roll
IRF_ITERATIONS = 300
CHANNEL = 12
C_REF = 1.0


def extract_channel_hist(data_6d, channel=CHANNEL):
    return np.asarray(data_6d[0, 0, :, :, :, channel].sum(axis=(0, 1)), dtype=float)


def build_time_axis(metadata):
    nbin = int(metadata.dfd_nbins)
    period_ns = 1 / (metadata.dfd_freq * 1e6) * 1e9
    dt_ns = period_ns / nbin
    t_ns = np.arange(nbin) * dt_ns
    return nbin, dt_ns, period_ns, t_ns


def circular_shift_bins(delta_ns, dt_ns, nbin):
    return float(np.mod(delta_ns / dt_ns, nbin))


def normalize_for_plot(hist):
    hist = np.asarray(to_numpy_1d(hist, dtype=float), dtype=float)
    return hist / hist.max()


def estimate_irf(reference_hist, t_ns, tau_ref_ns, period_ns, iterations=IRF_ITERATIONS):
    return np.asarray(
        to_numpy_1d(
            IRF_from_data_deconvolution(reference_hist, t_ns, C_REF, tau_ref_ns, period_ns, iterations=iterations),
            dtype=float,
        ),
        dtype=float,
    )


def initial_delay_from_phasor(data_hist, irf_hist, dt_ns, period_ns, nbin):
    _, _, delay_data_ns = phasor_delay_from_hist(data_hist, period_ns)
    _, _, delay_irf_ns = phasor_delay_from_hist(irf_hist, period_ns)
    delta_ns = delay_irf_ns - delay_data_ns
    delta_bins = circular_shift_bins(delta_ns, dt_ns, nbin)
    return float(delta_ns), float(delta_bins), float(delay_data_ns), float(delay_irf_ns)


def unwrap_circular_bins(values_bins, nbin, anchor_bins=None):
    values_bins = np.asarray(values_bins, dtype=float)
    unwrapped_bins = np.unwrap(values_bins / nbin * 2 * np.pi) / (2 * np.pi) * nbin
    if anchor_bins is not None and len(unwrapped_bins):
        unwrapped_bins = unwrapped_bins - nbin * np.round((unwrapped_bins[0] - anchor_bins) / nbin)
    return unwrapped_bins


def evaluate_fitted_histogram(t_ns, fit_result, irf_hist, period_ns):
    return np.asarray(
        to_numpy_1d(
            fit_model_data(
                t_ns,
                fit_result["C"],
                fit_result["dT"],
                fit_result["tau"],
                irf=irf_hist,
                period=period_ns,
            ),
            dtype=float,
        ),
        dtype=float,
    )


def run_fit(
    data_hist,
    ref_hist_raw,
    ref_roll_bins,
    t_ns,
    dt_ns,
    period_ns,
    nbin,
    tau_ref_ns,
    initial_dT_bins=None,
    initial_tau_ns=None,
    initial_C=None,
):
    ref_hist = np.roll(ref_hist_raw, ref_roll_bins)
    est_irf_ref = estimate_irf(ref_hist, t_ns, tau_ref_ns, period_ns)

    phasor_dT_ns, phasor_dT_bins, delay_data_ns, delay_irf_ns = initial_delay_from_phasor(
        data_hist,
        est_irf_ref,
        dt_ns,
        period_ns,
        nbin,
    )

    if initial_dT_bins is None:
        initial_dT_bins = phasor_dT_bins
    initial_dT_bins = circular_shift_bins(initial_dT_bins * dt_ns, dt_ns, nbin)
    initial_dT_ns = float(initial_dT_bins * dt_ns)

    fit_result, fit_cov = perform_fit_data(
        t_ns,
        data_hist,
        est_irf_ref,
        period_ns,
        initial_dT=initial_dT_bins,
        initial_tau=initial_tau_ns,
        initial_C=initial_C,
    )

    fitted_hist = evaluate_fitted_histogram(t_ns, fit_result, est_irf_ref, period_ns)

    return {
        "ref_roll_bins": int(ref_roll_bins),
        "ref_hist": ref_hist,
        "est_irf_ref": est_irf_ref,
        "fit_result": fit_result,
        "fit_cov": fit_cov,
        "fitted_hist": fitted_hist,
        "phasor_dT_ns": phasor_dT_ns,
        "phasor_dT_bins": phasor_dT_bins,
        "initial_dT_ns": initial_dT_ns,
        "initial_dT_bins": initial_dT_bins,
        "delay_data_ns": delay_data_ns,
        "delay_irf_ns": delay_irf_ns,
        "dT_fit_ns": float(fit_result["dT"] * dt_ns),
    }


# Single-Fit Workflow


This notebook now uses a single fitting path throughout: the binned periodic exponential model together with the circular fitter.


The workflow is:

1. Load one reference histogram and one data histogram.
2. Estimate the IRF from the reference decay.
3. Build a phasor-based initial delay guess.
4. Fit the data with `perform_fit_data(..., fit_type="circular")`.
5. Optionally repeat the same workflow for a scan of reference roll shifts.


In [ ]:
f_ref = "/mnt/DATA/Mixed Data/26-03-17_Convallaria_and_FLIMLABS_calibrated_Yellow_Slide/FLIMLABS_Yellow_slide_2_5ns-17-03-2026-16-18-22.h5"
tau_ref_ns = 2.5

f_data = "/mnt/DATA/Mixed Data/26-03-17_Convallaria_and_FLIMLABS_calibrated_Yellow_Slide/01_Convallaria_DFD_40MHz-17-03-2026-16-59-41.h5"

ref_data_6d, metadata_ref = mcs.load(f_ref)
data_6d, metadata_data = mcs.load(f_data)

ref_hist_raw = extract_channel_hist(ref_data_6d)
data_hist_raw = extract_channel_hist(data_6d)

nbin, dt_ns, period_ns, t_ns = build_time_axis(metadata_data)

print(f"channel = {CHANNEL}")
print(f"nbin = {nbin}")
print(f"dt_ns = {dt_ns:.6f} ns")
print(f"period_ns = {period_ns:.6f} ns")
print(f"reference counts = {ref_hist_raw.sum():.0f}")
print(f"data counts = {data_hist_raw.sum():.0f}")


In [ ]:
ref_roll_bins = 0
single_fit = run_fit(
    data_hist=data_hist_raw,
    ref_hist_raw=ref_hist_raw,
    ref_roll_bins=ref_roll_bins,
    t_ns=t_ns,
    dt_ns=dt_ns,
    period_ns=period_ns,
    nbin=nbin,
    tau_ref_ns=tau_ref_ns,
)

fit_result = single_fit["fit_result"]
fit_cov = single_fit["fit_cov"]
ref_hist = single_fit["ref_hist"]
est_irf_ref = single_fit["est_irf_ref"]
fitted_hist = single_fit["fitted_hist"]

summary = pd.Series(
    {
        "model": "binned periodic exponential",
        "fitter": "circular",
        "ref_roll_bins": single_fit["ref_roll_bins"],
        "ref_roll_ns": single_fit["ref_roll_bins"] * dt_ns,
        "initial_dT_bins": single_fit["initial_dT_bins"],
        "initial_dT_ns": single_fit["initial_dT_ns"],
        "C_fit_norm": fit_result["C"],
        "dT_fit_bins": fit_result["dT"],
        "dT_fit_ns": single_fit["dT_fit_ns"],
        "tau_fit_ns": fit_result["tau"],
    },
    name="value",
)

display(summary.to_frame())


# Plot The Single Fit


The reported `dT_fit_bins` is now the single public circular delay convention used everywhere in `Alignment`:
- it is measured in histogram bins,
- its canonical representation is `0 <= dT_fit_bins < nbin`,
- positive values follow the same sign as a positive roll applied to the reference / IRF histogram,
- the returned value can be passed directly to `fit_model_data` for plotting,
- the fitter handles the internal circular branch locally around the initial guess, so the `45.5` region is no longer a hard clipping point.


In [ ]:
irf_hist = np.asarray(hist_for_plot(est_irf_ref), dtype=float)
data_hist_plot = np.asarray(hist_for_plot(data_hist_raw), dtype=float)
fitted_hist_plot = np.asarray(hist_for_plot(fitted_hist), dtype=float)

phasor_irf, phase_irf_rad, delay_irf_ns = phasor_delay_from_hist(irf_hist, period_ns)
phasor_data, phase_data_rad, delay_data_ns = phasor_delay_from_hist(data_hist_plot, period_ns)
phasor_fit, phase_fit_rad, delay_fit_hist_ns = phasor_delay_from_hist(fitted_hist_plot, period_ns)
delta_delay_ns = np.mod(delay_data_ns - delay_irf_ns, period_ns)

title_text = (
    "Single fit summary\n"
    + f"phasor delta = {delta_delay_ns:.3f} ns\n"
    + f"fit shift = {single_fit['dT_fit_ns']:.3f} ns, tau = {fit_result['tau']:.3f} ns"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

theta = np.linspace(0, 2 * np.pi, 400)
axes[0].plot(np.cos(theta), np.sin(theta), '--', color='0.8', label='unit circle')
axes[0].quiver(0, 0, np.real(phasor_irf), np.imag(phasor_irf), angles='xy', scale_units='xy', scale=1, color='tab:blue', label='IRF phasor')
axes[0].quiver(0, 0, np.real(phasor_data), np.imag(phasor_data), angles='xy', scale_units='xy', scale=1, color='tab:orange', label='Data phasor')
axes[0].quiver(0, 0, np.real(phasor_fit), np.imag(phasor_fit), angles='xy', scale_units='xy', scale=1, color='tab:red', label='Fitted histogram phasor')
axes[0].scatter(
    [np.real(phasor_irf), np.real(phasor_data), np.real(phasor_fit)],
    [np.imag(phasor_irf), np.imag(phasor_data), np.imag(phasor_fit)],
    color=['tab:blue', 'tab:orange', 'tab:red'],
)
axes[0].set_xlabel('g')
axes[0].set_ylabel('s')
axes[0].set_title('Phasor comparison')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(t_ns, normalize_for_plot(ref_hist), label='Reference histogram', color='tab:blue', alpha=0.6)
axes[1].plot(t_ns, normalize_for_plot(est_irf_ref), label='Estimated IRF', color='tab:green')
axes[1].plot(t_ns, normalize_for_plot(data_hist_raw), label='Data histogram', color='black')
axes[1].plot(t_ns, normalize_for_plot(fitted_hist), label='Fitted histogram', color='tab:red')
axes[1].axvline(delay_irf_ns, linestyle='--', color='tab:blue', label=f'IRF delay = {delay_irf_ns:.3f} ns')
axes[1].axvline(delay_data_ns, linestyle='--', color='tab:orange', label=f'Data delay = {delay_data_ns:.3f} ns')
axes[1].axvline(delay_fit_hist_ns, linestyle='--', color='tab:red', label=f'Fitted histogram delay = {delay_fit_hist_ns:.3f} ns')
axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('Normalized intensity')
axes[1].set_title(title_text)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


# Reference-Roll Scan


In [ ]:
roll_shift_bins_values = np.arange(nbin)
data_hist_plot = np.asarray(hist_for_plot(data_hist_raw), dtype=float)
_, _, delay_data_ns_plot = phasor_delay_from_hist(data_hist_plot, period_ns)

scan_rows = []
previous_fit_result = None
previous_roll_shift_bins = None

for roll_shift_bins in tqdm(roll_shift_bins_values):
    if previous_fit_result is None:
        initial_dT_bins_override = None
        initial_tau_ns_override = None
        initial_C_override = None
    else:
        roll_step_bins = int(roll_shift_bins - previous_roll_shift_bins)
        initial_dT_bins_override = np.mod(previous_fit_result["dT"] + roll_step_bins, nbin)
        initial_tau_ns_override = previous_fit_result["tau"]
        initial_C_override = previous_fit_result["C"]

    fit_run = run_fit(
        data_hist=data_hist_raw,
        ref_hist_raw=ref_hist_raw,
        ref_roll_bins=roll_shift_bins,
        t_ns=t_ns,
        dt_ns=dt_ns,
        period_ns=period_ns,
        nbin=nbin,
        tau_ref_ns=tau_ref_ns,
        initial_dT_bins=initial_dT_bins_override,
        initial_tau_ns=initial_tau_ns_override,
        initial_C=initial_C_override,
    )

    fit_result_loop = fit_run["fit_result"]
    previous_fit_result = fit_result_loop
    previous_roll_shift_bins = roll_shift_bins
    fit_cov_loop = fit_run["fit_cov"]
    perr = np.sqrt(np.diag(fit_cov_loop))

    irf_hist_loop = np.asarray(hist_for_plot(fit_run["est_irf_ref"]), dtype=float)
    _, _, delay_irf_ns_loop = phasor_delay_from_hist(irf_hist_loop, period_ns)
    phasor_delta_ns_loop = np.mod(delay_irf_ns_loop - delay_data_ns_plot, period_ns)

    scan_rows.append(
        {
            "roll_shift_bins": roll_shift_bins,
            "roll_shift_ns": roll_shift_bins * dt_ns,
            "phasor_dT_bins": fit_run["phasor_dT_bins"],
            "phasor_dT_ns": fit_run["phasor_dT_ns"],
            "initial_dT_bins": fit_run["initial_dT_bins"],
            "initial_dT_ns": fit_run["initial_dT_ns"],
            "C_fit_norm": fit_result_loop["C"],
            "C_fit_perr_norm": perr[0],
            "dT_fit_bins": fit_result_loop["dT"],
            "dT_fit_ns": fit_run["dT_fit_ns"],
            "dT_fit_perr_bins": perr[1] / dt_ns,
            "dT_fit_perr_ns": perr[1],
            "tau_fit_ns": fit_result_loop["tau"],
            "tau_fit_perr_ns": perr[2],
            "phasor_delta_ns": phasor_delta_ns_loop,
        }
    )

scan_results_df = pd.DataFrame(scan_rows)
scan_results_df["dT_fit_bins_unwrapped"] = unwrap_circular_bins(scan_results_df["dT_fit_bins"], nbin)
scan_results_df["dT_fit_ns_unwrapped"] = scan_results_df["dT_fit_bins_unwrapped"] * dt_ns
scan_results_df["phasor_dT_bins_unwrapped"] = unwrap_circular_bins(
    scan_results_df["phasor_dT_bins"],
    nbin,
    anchor_bins=scan_results_df["dT_fit_bins_unwrapped"].iloc[0],
)
scan_results_df["phasor_dT_ns_unwrapped"] = scan_results_df["phasor_dT_bins_unwrapped"] * dt_ns
scan_results_df["initial_dT_bins_unwrapped"] = unwrap_circular_bins(
    scan_results_df["initial_dT_bins"],
    nbin,
    anchor_bins=scan_results_df["dT_fit_bins_unwrapped"].iloc[0],
)
scan_results_df["initial_dT_ns_unwrapped"] = scan_results_df["initial_dT_bins_unwrapped"] * dt_ns

display(
    scan_results_df[
        [
            "roll_shift_bins",
            "roll_shift_ns",
            "phasor_dT_bins",
            "phasor_dT_ns",
            "initial_dT_bins",
            "initial_dT_ns",
            "dT_fit_bins",
            "dT_fit_ns",
            "dT_fit_ns_unwrapped",
            "tau_fit_ns",
            "phasor_delta_ns",
        ]
    ].round(6)
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), gridspec_kw={"width_ratios": [1.2, 1.0, 1.0]})

axes[0].plot(scan_results_df["roll_shift_ns"], scan_results_df["initial_dT_ns"], 'o--', label='Initial delay')
axes[0].plot(scan_results_df["roll_shift_ns"], scan_results_df["dT_fit_ns"], 'o-', label='Fit delay')
axes[0].plot(scan_results_df["roll_shift_ns"], scan_results_df["phasor_dT_ns"], 's--', label='Phasor delay')
axes[0].set_xlabel('Applied reference roll (ns)')
axes[0].set_ylabel('Delay (ns)')
axes[0].set_title('Delay vs reference roll')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(scan_results_df["roll_shift_ns"], scan_results_df["tau_fit_ns"], 'o-', color='tab:green')
axes[1].set_xlabel('Applied reference roll (ns)')
axes[1].set_ylabel('Fitted tau (ns)')
axes[1].set_title('Lifetime stability')
axes[1].grid(True, alpha=0.3)

axes[2].plot(
    scan_results_df["roll_shift_ns"],
    scan_results_df["phasor_dT_ns"] - scan_results_df["dT_fit_ns"],
    'o-',
    color='tab:purple',
)
axes[2].axhline(0.0, color='0.5', linestyle='--')
axes[2].set_xlabel('Applied reference roll (ns)')
axes[2].set_ylabel('Phasor - fit delay (ns)')
axes[2].set_title('Delay mismatch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
scan_results_df[["dT_fit_bins", "dT_fit_ns", "dT_fit_ns_unwrapped", "tau_fit_ns", "phasor_dT_ns"]].describe().round(6)
